### Writing Viterbi Algorithm for the Primer (3 marks)

Importing Library

In [1]:
import math

Defining states and probabilities

In [2]:
states = ['E', 'I' , '5'] 

trans_probs = {
    'start': {'E': 1.0, 'I': 0.0, '5': 0.0, 'end': 0.0, 'start': 0.0},
    'E':    {'E': 0.9, '5' : 0.1, 'I': 0.0, 'end': 0.0 , 'start': 0.0},
    'I':    {'E': 0.0, '5' : 0.0, 'I': 0.9, 'end': 0.1 , 'start': 0.0},
    '5':    {'E': 0.0, '5' : 0.0, 'I': 1.0, 'end': 0.0 , 'start': 0.0},
    'end':  {'E': 0.0, '5' : 0.0, 'I': 0.0, 'end': 1.0 , 'start': 0.0},
}

emission_probs = {
    'E': {'A': 0.25 , 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5' : {'A': 0.05 , 'C': 0 ,'G': 0.95 , 'T': 0 },
    'I': {'A': 0.4 , 'C': 0.1 , 'G': 0.1 , 'T': 0.4 },
}

Custom log function to handle log(0)

In [3]:
def log(x: float) -> float:
    if x == 0.0:
        return float('-inf')
    else:
        return math.log(x)

Function to compute log probability for given sequence and path

In [4]:
def get_log_prob_of_a_given_path(path: str, seq: str) -> float:
    
    if len(path) != len(seq):
        raise ValueError("Path and sequence must have same length.")
    
    lp = log(trans_probs['start'][path[0]])  

    for i in range(0, len(path)-1):
        cur_s = path[i]
        next_s = path[i+1]
        lp += log(emission_probs[cur_s][seq[i]])
        lp += log(trans_probs[cur_s][next_s])

    lp += log(emission_probs[path[-1]][seq[-1]])
    lp += log(trans_probs[path[-1]]['end'])

    return round(lp, 2)

Viterbi algorithm to find the most likely state path for given sequence

In [5]:
def viterbi(obs_seq: str):

    n = len(obs_seq)
    V = [{} for _ in range(n)]
    backpointer = [{} for _ in range(n)]

    # Build the Viterbi table

    for s in states:
        V[0][s] = log(trans_probs['start'][s]) + log(emission_probs[s][obs_seq[0]])
        backpointer[0][s] = 'start'

    for t in range(1, n):

        for s in states:

            max_prob, prev_s = float('-inf'), None
            for sp in states:
                prob = V[t-1][sp] + log(trans_probs[sp][s])
                if prob > max_prob:
                    max_prob, prev_s = prob, sp

            V[t][s] = max_prob + log(emission_probs[s][obs_seq[t]])
            backpointer[t][s] = prev_s

    # backtrack path
    path = []

    max_final, last_state = float('-inf'), None    
    for s in states:
        if V[n-1][s] > max_final:
            max_final, last_state = V[n-1][s], s
    path.insert(0, last_state)

    for t in range(n-1, 0, -1):
        path.insert(0, backpointer[t][path[0]])

    # convert path to string
    path = ''.join(path)
    
    return path, round(max_final, 2)

Testing on given sequence

In [6]:
# Example 1 : finding the most likely path for a given sequence

ex_seq = "CTTCATGTGAAAGCAGACGTAAGTCA"
ex_path, ex_prob = viterbi(ex_seq)
print("Most likely path:", ex_path)

Most likely path: EEEEEEEEEEEEEEEEEEEEEEEEEE


In [7]:
# Example 2 : finding the log-probability of a given path for a given sequence

seq = "CTTCATGTGAAAGCAGACGTAAGTCA"
path = "EEEEEEEEEEEEEEEEEE5IIIIIII"

print(get_log_prob_of_a_given_path(path, seq))

-41.22
